# Démo : Reconstruction 3D + Segmentation Sémantique d'une scène de rue

Ce notebook illustre l'utilisation des modules du projet :
1. Segmentation sémantique 2D (DeepLabV3 + SAM)
2. Reconstruction 3D (Open3D, RGB-D)
3. Fusion 2D -> 3D et visualisation

Placez vos données dans `../data/color/` et `../data/depth/` avant d'exécuter.

In [9]:
#!pip install open3d
!pip install -r ../requirements.txt

  Cloning https://github.com/facebookresearch/segment-anything.git to c:\users\hp\appdata\local\temp\pip-install-t3oiz6i5\segment-anything_d059dfd143274e39b96ab00c522da4ca
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/241.2 MB ? eta -:--:--
   ---------------------------------------- 0.5/241.2 MB 5.6 MB/s eta 0:00:44
   ---------------------------------------- 2.1/241.2 MB 6.2 MB/s eta 0:00:39
    -----------------------

  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git 'C:\Users\HP\AppData\Local\Temp\pip-install-t3oiz6i5\segment-anything_d059dfd143274e39b96ab00c522da4ca'

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import sys
sys.path.append('../src')
#sys.path.append('/content/3d-street-reconstruction/src')

from pathlib import Path
import open3d as o3d

DATA_DIR = Path('../data')
OUTPUT_DIR = Path('../outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

import os
print(os.getcwd())  # doit afficher .../3d-street-reconstruction/notebooks

c:\Users\HP\Documents\Computer-Vision-Project\3D-street-reconstruction\notebooks


## 1. Segmentation sémantique d'une image

In [ ]:
from semantic_segmentation import segment_image
import matplotlib.pyplot as plt
from PIL import Image

#sample_image = sorted((DATA_DIR / 'color').glob('*'))[0]
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
images = sorted([p for p in (DATA_DIR / 'color').glob('*') if p.suffix.lower() in VALID_EXTENSIONS])
if not images:
    raise FileNotFoundError("Aucune image trouvée dans data/color/ — ajoutes-en !")
sample_image = images[0]

out_path = OUTPUT_DIR / 'segmentations' / f'{sample_image.stem}.png'
out_path.parent.mkdir(parents=True, exist_ok=True)

supercat_map, color_map = segment_image(sample_image, out_path, use_sam=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(Image.open(sample_image))
axes[0].set_title('Image originale')
axes[0].axis('off')
axes[1].imshow(color_map)
axes[1].set_title('Segmentation sémantique')
axes[1].axis('off')
plt.show()

UnidentifiedImageError: cannot identify image file '..\\data\\color\\.gitkeep'

## 2. Reconstruction 3D de la scène

In [ ]:
from reconstruction import build_scene_from_rgbd_sequence

scene_pcd = build_scene_from_rgbd_sequence(DATA_DIR / 'color', DATA_DIR / 'depth', voxel_size=0.03)
scene_path = OUTPUT_DIR / 'scene.ply'
o3d.io.write_point_cloud(str(scene_path), scene_pcd)
print(f'{len(scene_pcd.points)} points reconstruits')

## 3. Fusion sémantique 2D -> 3D

Exécutez d'abord la segmentation sur toutes les frames (ou utilisez `pipeline.py`), puis :

In [ ]:
from fusion_2d_3d import fuse_sequence

semantic_path = OUTPUT_DIR / 'scene_semantic.ply'
scene_semantic = fuse_sequence(
    DATA_DIR / 'color', DATA_DIR / 'depth',
    OUTPUT_DIR / 'segmentations', semantic_path, voxel_size=0.03
)

## 4. Visualisation 3D

La visualisation Open3D s'ouvre dans une fenêtre séparée (ne fonctionne pas dans tous les environnements notebook).

In [ ]:
o3d.visualization.draw_geometries([scene_semantic])